In [1]:
import tensorflow as tf
import numpy as np
import os

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight

In [2]:
TRAIN_DIR = "dataset/Training"
VAL_DIR = "dataset/Validation"

IMG_SIZE = 224
BATCH_SIZE = 32

In [3]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("Classes:", train_data.class_indices)

Found 7384 images belonging to 2 classes.
Found 985 images belonging to 2 classes.
Classes: {'bleached_corals': 0, 'healthy_corals': 1}


In [4]:
classes = train_data.classes
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),
    y=classes
)

class_weight = dict(enumerate(weights))
print("Class weights:", class_weight)

Class weights: {0: np.float64(0.9515463917525773), 1: np.float64(1.0536529680365296)}


In [6]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze most layers
for layer in base_model.layers[:-50]:
    layer.trainable = False

# Unfreeze last 50 layers
for layer in base_model.layers[-50:]:
    layer.trainable = True

# Custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# Compile
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,378,021 (16.70 MB)

 Trainable params: 2,855,314 (10.89 MB)

 Non-trainable params: 1,522,707 (5.81 MB)

In [7]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    class_weight=class_weight
)

Epoch 1/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 121s 488ms/step - accuracy: 0.7239 - loss: 0.5274 - val_accuracy: 0.8579 - val_loss: 0.3545
Epoch 2/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 109s 472ms/step - accuracy: 0.8319 - loss: 0.3700 - val_accuracy: 0.8934 - val_loss: 0.2506
Epoch 3/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 119s 517ms/step - accuracy: 0.8747 - loss: 0.2827 - val_accuracy: 0.9310 - val_loss: 0.1796
Epoch 4/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 123s 534ms/step - accuracy: 0.9082 - loss: 0.2205 - val_accuracy: 0.9503 - val_loss: 0.1302
Epoch 5/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 126s 543ms/step - accuracy: 0.9307 - loss: 0.1756 - val_accuracy: 0.9513 - val_loss: 0.1233
Epoch 6/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 123s 533ms/step - accuracy: 0.9399 - loss: 0.1445 - val_accuracy: 0.9716 - val_loss: 0.0806
Epoch 7/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 125s 542ms/step - accuracy: 0.9472 - loss: 0.1289 - val_accuracy: 0.9766 - val_loss: 0.0682
Epoch 8/10
231/231 ━━━━━━━━━━━━━━━━━━━━ 131s 566ms/step - accuracy: 0.9575 -

### Validate the Model

In [13]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

val_data = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # 🔥 THIS FIXES YOUR PROBLEM
)

Found 985 images belonging to 2 classes.


In [14]:
val_data.reset()

In [15]:
import numpy as np

predictions = model.predict(val_data)
y_pred = np.argmax(predictions, axis=1)
y_true = val_data.classes

31/31 ━━━━━━━━━━━━━━━━━━━━ 11s 344ms/step


In [16]:
from sklearn.metrics import classification_report, confusion_matrix

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))


Classification Report:

              precision    recall  f1-score   support

           0       0.97      1.00      0.98       485
           1       1.00      0.97      0.98       500

    accuracy                           0.98       985
   macro avg       0.98      0.98      0.98       985
weighted avg       0.98      0.98      0.98       985


Confusion Matrix:

[[483   2]
 [ 15 485]]
